# 01 · ROS 2: a graph you can observe
**15 minutes · first notebook.** Objective: distinguish a node, a typed topic, a publisher and a subscriber by actually running each.

Use **Run → Run Selected Cell** and read the result before advancing. The introduction slides belong to your instructor; this notebook does not replace them.

We use an isolated, CPU-only ROS 2 Lyrical container, domain **71**, and `/notebook_course/chatter`. These are ordinary `std_msgs/String` messages, **not robot sensor data**. The native simulator, ROS gateway and robot are not started here. Our host Jupyter interpreter must never import `rclpy`, Isaac Sim or Kit.

In [ ]:
from pathlib import Path
import os, sys, json
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the host Python 3.12 kernel created by scripts/notebook_setup.sh")
# This kernel is deliberately not a ROS / Kit interpreter.
start = Path(os.environ.get("WORKSHOP_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "notebooks/workshop.py").is_file()
             and (p / "src/microduck_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository, or set WORKSHOP_ROOT")
sys.path.insert(0, str(ROOT / "notebooks"))
from workshop import Session
session = Session(ROOT, work=os.environ.get("WORKSHOP_NOTEBOOK_WORK"))
VALIDATE = os.environ.get("WORKSHOP_NOTEBOOK_MODE") == "validate"
print("Host Python:", sys.version.split()[0])
print("Mode:", "safe validation (hardware SKIPPED)" if VALIDATE else "interactive")

## Predict · 2 minutes
If there is one publisher but no subscriber, does a topic still exist? What does its **type** tell you that its name does not?

Set `RUN_ROS = True` after reading the command. It needs Docker and the prebuilt workshop ROS image. Nothing is pulled or built automatically. Set the image variable to the instructor's prepared tag if different.

In [ ]:
RUN_ROS = os.environ.get("WORKSHOP_NOTEBOOK_MODE") == "cpu"  # otherwise change to True
ROS_IMAGE = os.environ.get("WORKSHOP_ROS_IMAGE", "roscon/microduck-ros5:local")
TOPIC = "/notebook_course/chatter"
RATE_HZ = 2
TEXT = "hello from ROS 2"

In [ ]:
def start_tutorial():
    session.run(["docker", "info", "--format", "{{.ServerVersion}}"], timeout=15)
    session.run(["docker", "image", "inspect", ROS_IMAGE, "--format", "{{.Id}}"], timeout=15)
    session.start("tutorial", ROS_IMAGE,
        ["ros2", "topic", "pub", TOPIC, "std_msgs/msg/String", "{data: '" + TEXT + "'}",
         "--rate", str(RATE_HZ), "--node-name", "tutorial_publisher"])
    session.wait_ros_node("tutorial", "/tutorial_publisher", timeout=30)
    session.ros("tutorial", ["topic", "echo", TOPIC, "std_msgs/msg/String", "--once"], timeout=15)
session.step("start isolated real ROS publisher", RUN_ROS, start_tutorial)

## Observe · 4 minutes
`node list` describes participants. `topic list -t` describes channels and types. `topic info --verbose` exposes endpoints and QoS, which must be compatible for communication.

The CLI subscriber below is a real node while `echo` runs. It disappears when `--once` completes. The ROS daemon is a graph helper, not the source of your message.

In [ ]:
def inspect_graph():
    session.ros("tutorial", ["node", "list"])
    session.ros("tutorial", ["node", "info", "/tutorial_publisher"])
    session.ros("tutorial", ["topic", "list", "-t"])
    session.ros("tutorial", ["topic", "info", TOPIC, "--verbose"])
    session.ros("tutorial", ["interface", "show", "std_msgs/msg/String"])
session.step("inspect actual ROS graph", RUN_ROS, inspect_graph)

## Task · 4 minutes
Read one message. Then change `EXTRA_TEXT` below and publish a **second** message, deliberately using the same type. Predict whether an `echo --once` subscriber can guarantee it catches your one-off message while another publisher is active. It cannot: discovery and timing matter.

The second publisher uses a bounded subscriber-discovery wait, so a missing subscriber cannot hang the lesson. Compare endpoint counts before and after; do not treat a timeout as a successful exchange.

In [ ]:
EXTRA_TEXT = "I can name the message type"
def read_and_publish():
    result = session.ros("tutorial", ["topic", "echo", TOPIC, "std_msgs/msg/String", "--once"])
    if "data:" not in result.stdout:
        raise RuntimeError("No actual message received")
    session.ros("tutorial", ["topic", "pub", TOPIC, "std_msgs/msg/String",
        json.dumps({"data": EXTRA_TEXT}), "--once", "--wait-matching-subscriptions", "0",
        "--node-name", "student_publisher"])
    session.complete("01", {"kind": "CPU_TUTORIAL", "topic": TOPIC,
                            "message_type": "std_msgs/msg/String"})
session.step("actual subscriber and student publisher", RUN_ROS, read_and_publish)

## Debug · 2 minutes
- **Docker missing/permission denied:** use the prepared machine and ask staff; do not `sudo pip install` ROS into this kernel.
- **Image missing:** staff must build the documented pinned image before class. A Docker tag alone is not a runtime version proof.
- **No message:** inspect the publisher log below, domain 71, exact topic/type and QoS. Do not inject fake joint states.
- **Container already exists after interruption:** run the cleanup cell, then explicitly restart. Kernel restart does not stop Docker.

**Checkpoint:** explain the four graph concepts in your own words. The saved checkpoint records only this CPU tutorial, not a robot run.

## Cleanup · 3 minutes
This removes only the labeled container created by this notebook session; logs remain private. Never run `docker compose down` or broad process-kill commands on a shared machine. Next: [Isaac ROS 5](02_isaac_ros_5.ipynb).

In [ ]:
session.step("owned tutorial cleanup", RUN_ROS, lambda: session.stop("tutorial"))